In [ ]:
# --- Imports ---
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms as T
from PIL import Image
import numpy as np
import pickle
from tqdm import tqdm
import random

# --- Constants ---
TRAIN_ROOT = "data/train"
BATCH_SIZE = 32
IMAGE_SIZE = 128
EPOCHS = 10
SSL_WEIGHT = 0.5
FSL_WEIGHT = 0.5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
HYBRID_FEATURES_PKL = "hybrid_features.pkl"
HYBRID_ENCODER_PTH = "hybrid_encoder.pth"
PROJECTION_DIM = 128
OUT_DIM = 512

# --- Data Transform ---
train_transform = T.Compose([
    T.RandomResizedCrop(IMAGE_SIZE),
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.4, 0.4, 0.4, 0.1),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

extract_transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

# --- Dataset for SSL ---
class SSLDataset(Dataset):
    def __init__(self, root, transform=None):
        self.samples = []
        self.transform = transform
        class_to_idx = {"real": 0, "fake": 1}  # Map class names to numeric labels

        for cls in os.listdir(root):
            cls_path = os.path.join(root, cls)
            if not os.path.isdir(cls_path):
                continue
            label = class_to_idx.get(cls, None)
            if label is None:
                continue
            for img in os.listdir(cls_path):
                img_path = os.path.join(cls_path, img)
                if img_path.lower().endswith((".png", ".jpg", ".jpeg")):
                    self.samples.append((img_path, label))
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        img1 = self.transform(img)
        img2 = self.transform(img)
        return img1, img2, label

# --- Simple Episodic Sampler for FSL ---
class FewShotSampler:
    def __init__(self, root):
        self.root = root
        self.class_to_imgs = {}
        for cls in os.listdir(root):
            cls_path = os.path.join(root, cls)
            if not os.path.isdir(cls_path):
                continue
            imgs = [os.path.join(cls_path, x) for x in os.listdir(cls_path)
                    if x.lower().endswith((".png", ".jpg", ".jpeg"))]
            self.class_to_imgs[cls] = imgs
        self.classes = list(self.class_to_imgs.keys())

    def sample_episode(self, n_way=2, k_shot=5, q_query=5):
        chosen = random.sample(self.classes, n_way)
        support, query, support_labels, query_labels = [], [], [], []
        for i, c in enumerate(chosen):
            imgs = random.sample(self.class_to_imgs[c], k_shot + q_query)
            support += imgs[:k_shot]
            query += imgs[k_shot:]
            support_labels += [i] * k_shot
            query_labels += [i] * q_query
        return support, query, support_labels, query_labels


# --- Hybrid Encoder (Shared Backbone) ---
class HybridEncoder(nn.Module):
    def __init__(self, out_dim=OUT_DIM, proj_dim=PROJECTION_DIM):
        super().__init__()
        base = models.resnet18(weights=None)
        self.encoder = nn.Sequential(*list(base.children())[:-1])
        in_features = base.fc.in_features
        self.projection = nn.Sequential(
            nn.Linear(in_features, out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, proj_dim)
        )
        self.out_dim = out_dim
    def forward(self, x):
        x = self.encoder(x)
        x = x.view(x.size(0), -1)
        z = self.projection(x)
        return x, F.normalize(z, dim=1)

# --- SimCLR Loss ---
def simclr_loss(z_i, z_j, temperature=0.5):
    batch_size = z_i.shape[0]
    z = torch.cat([z_i, z_j], dim=0)
    sim = F.cosine_similarity(z.unsqueeze(1), z.unsqueeze(0), dim=2)
    mask = torch.eye(2 * batch_size, dtype=torch.bool).to(z.device)
    sim = sim[~mask].view(2 * batch_size, -1)
    positives = torch.cat([torch.arange(batch_size) + batch_size,
                           torch.arange(batch_size)], dim=0).to(z.device)
    pos_sim = F.cosine_similarity(z, z[positives])
    numerator = torch.exp(pos_sim / temperature)
    denominator = torch.sum(torch.exp(sim / temperature), dim=1)
    loss = -torch.log(numerator / denominator)
    return loss.mean()

# --- ProtoNet Loss ---
def prototypical_loss(encoder, support, query, y_s, y_q):
    xs, _ = encoder(support)
    xq, _ = encoder(query)
    n_way = len(torch.unique(y_s))
    prototypes = torch.stack([xs[y_s == c].mean(0) for c in range(n_way)])
    dists = torch.cdist(xq, prototypes)
    log_p_y = F.log_softmax(-dists, dim=1)
    loss = F.nll_loss(log_p_y, y_q)
    _, y_hat = log_p_y.max(1)
    acc = torch.eq(y_hat, y_q).float().mean().item()
    return loss, acc

# --- Combined Hybrid Trainer ---
def train_hybrid_model(model, ssl_loader, fsl_sampler, epochs=EPOCHS, device=DEVICE):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-4)
    model.train()
    for epoch in range(1, epochs + 1):
        ssl_iter = iter(ssl_loader)
        total_loss, total_acc = 0, 0
        for _ in tqdm(range(len(ssl_loader)), desc=f"Hybrid Epoch {epoch}/{epochs}"):
            try:
                x1, x2, _ = next(ssl_iter)
            except StopIteration:
                ssl_iter = iter(ssl_loader)
                x1, x2, _ = next(ssl_iter)

            # SSL Loss
            x1, x2 = x1.to(device), x2.to(device)
            _, z1 = model(x1)
            _, z2 = model(x2)
            loss_ssl = simclr_loss(z1, z2)

            # FSL Loss
            support, query, y_s, y_q = fsl_sampler.sample_episode()
            s_imgs = torch.stack([extract_transform(Image.open(p).convert("RGB")) for p in support]).to(device)
            q_imgs = torch.stack([extract_transform(Image.open(p).convert("RGB")) for p in query]).to(device)
            y_s = torch.tensor(y_s).to(device)
            y_q = torch.tensor(y_q).to(device)
            loss_fsl, acc = prototypical_loss(model, s_imgs, q_imgs, y_s, y_q)

            # Hybrid Loss
            loss = SSL_WEIGHT * loss_ssl + FSL_WEIGHT * loss_fsl
            opt.zero_grad()
            loss.backward()
            opt.step()

            total_loss += loss.item()
            total_acc += acc

        print(f"Epoch {epoch}: Loss={total_loss/len(ssl_loader):.4f} | FSL Acc={total_acc/len(ssl_loader):.4f}")

    torch.save(model.state_dict(), HYBRID_ENCODER_PTH)
    print(f"Hybrid encoder saved to {HYBRID_ENCODER_PTH}")
    return model

# --- Feature Extraction ---
def extract_hybrid_features(model, folder, transform, device, out_pkl):
    model.eval()
    features, labels = [], []
    with torch.no_grad():
        for cls in os.listdir(folder):
            cls_path = os.path.join(folder, cls)
            if not os.path.isdir(cls_path):
                continue
            for img_name in os.listdir(cls_path):
                img_path = os.path.join(cls_path, img_name)
                img = Image.open(img_path).convert("RGB")
                x = transform(img).unsqueeze(0).to(device)
                f, _ = model(x)
                features.append(f.cpu().numpy().squeeze())
                labels.append(0 if cls == "real" else 1)  # ✅ fixed
    features = np.array(features)
    labels = np.array(labels)
    with open(out_pkl, "wb") as f:
        pickle.dump({"features": features, "labels": labels}, f)
    print(f"Saved hybrid features to {out_pkl}")
    return features, labels

# --- Main Orchestration ---
if __name__ == "__main__":
    ssl_dataset = SSLDataset(TRAIN_ROOT, train_transform)
    ssl_loader = DataLoader(ssl_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    fsl_sampler = FewShotSampler(TRAIN_ROOT)

    model = HybridEncoder(out_dim=OUT_DIM, proj_dim=PROJECTION_DIM)
    model = train_hybrid_model(model, ssl_loader, fsl_sampler, epochs=EPOCHS, device=DEVICE)
    extract_hybrid_features(model, TRAIN_ROOT, extract_transform, DEVICE, HYBRID_FEATURES_PKL)


Hybrid Epoch 1/10:  41%|████      | 13/32 [01:07<01:37,  5.15s/it]


KeyboardInterrupt: 